# opencode serve qua Tailscale Funnel

Chạy **opencode serve** ngay trong Colab và truy cập từ bất kỳ đâu trên internet qua một Funnel URL công khai — không cần VPN. Một cell duy nhất, zero-config.

**Trước khi chạy:** mở sidebar trái **🔑 Secrets** và thêm credential của bạn với tên **chính xác**:
- `TS_CLIENT_SECRET` (OAuth trust credential `tskey-client-…`, khuyến nghị), **hoặc**
- `TS_AUTH_KEY` (`tskey-auth-…`), **hoặc** `TS_OAUTH_CLIENT_ID` + `TS_OAUTH_CLIENT_SECRET`, **hoặc** `TS_ACCESS_TOKEN` / `TS_API_KEY`

Rồi chạy cell bên dưới. Chi tiết: https://github.com/ongtrieuphuchieu689-7u/tailscale-cli/blob/main/examples/colab/README.md

In [ ]:
# One-cell Colab launcher for examples/colab/opencode-funnel-colab.sh
#
# Loads the Tailscale credential from the Colab Secrets panel (key icon)
# and runs the funnel script (fetched from the repo, always the latest).
#
# Why a Python cell? userdata.get() talks to the Colab UI over the kernel
# message channel; a bash/python3 subprocess has no such channel and times
# out ("Secrets can only be fetched when running from the Colab UI"). So the
# secrets must be read here, in the kernel, and handed to bash via the env.

from google.colab import userdata
import os, subprocess, urllib.request

for name in ("TS_AUTH_KEY", "TS_CLIENT_SECRET", "TS_OAUTH_CLIENT_ID",
             "TS_OAUTH_CLIENT_SECRET", "TS_ACCESS_TOKEN", "TS_API_KEY",
             "TS_TAILNET"):
    try:
        os.environ[name] = userdata.get(name)
    except Exception:
        pass  # secret not set -> skip

if not any(os.environ.get(k) for k in (
    "TS_AUTH_KEY", "TS_CLIENT_SECRET", "TS_OAUTH_CLIENT_SECRET",
    "TS_ACCESS_TOKEN", "TS_API_KEY")):
    print("ERROR: no Tailscale credential found. Add one to the Secrets "
          "panel (key icon, left sidebar) with an EXACT name - TS_CLIENT_SECRET, "
          "TS_AUTH_KEY, TS_OAUTH_CLIENT_SECRET, TS_ACCESS_TOKEN or TS_API_KEY - "
          "grant this notebook access, then re-run this cell.")
    raise SystemExit(1)

script = urllib.request.urlopen(
    "https://raw.githubusercontent.com/ongtrieuphuchieu689-7u/tailscale-cli/main/examples/colab/opencode-funnel-colab.sh"
).read().decode()
subprocess.run(["/bin/bash", "-c", script], check=True)

## Sau khi chạy thành công

Cell in ra Funnel URL dạng `https://colab-opencode.<tailnet>.ts.net/` — mở trên trình duyệt của bất kỳ thiết bị nào (không cần VPN; đợi ~1–2 phút nếu cert chưa cấp xong). Trả về 401 nghĩa là serve đang chạy đúng (cần đặt `OPENCODE_SERVER_PASSWORD` nếu muốn bảo vệ bằng user/pass).

Muốn dừng serve + daemon tailscaled, chạy cell bên dưới (tuỳ chọn).

In [ ]:
%%bash
tailscale-cli-opencode --stop 2>&1 || echo "nothing to stop (already stopped)"

## Ghi chú

- Script tự cài Node 22+ và `tailsacle-cli` (npm), rồi giao toàn bộ cho **một lệnh** `tailscale-cli-opencode --port 3000 --install --yes --apply-policy --enable-https --json` — resolve/install opencode qua `npx -y opencode-ai`, ghi permission config, chạy serve nền, join tailnet, publish Funnel ở port 443, auto-provision (tagOwners / funnel attribute / HTTPS), verify DNS + TLS trước khi in URL.
- Log serve: `~/.cache/tailsacle-cli/bin/opencode-serve.log`.
- Chạy lại cell chính trong session mới sẽ join một node mới (`colab-opencode-2`, `-3`, …) và dừng node cũ.
- Credential `userdata.get()` chỉ hoạt động trong Python cell (kênh kernel tới Colab UI) — vì thế launcher là Python cell, không phải `%%bash`.